# Løsningsforslag: Statistikk og dataanalyse (kap. 13–18)

Denne notatboken inneholder fullstendige Python-beregninger for:

- **Oppgave 14.3** (konfidensintervaller)
- **Oppgave 15.1, 15.2, 15.3** (hypotesetesting: grunnleggende teori)
- **Oppgave 16.4, 16.5, 16.6** (inferens for et gjennomsnitt)
- **Oppgave 17.5, 17.6** (inferens for en andel)
- **Oppgave 18.1, 18.2** (inferens for å sammenligne to grupper)

Alle numeriske svar er beregnet med `numpy` og `scipy.stats`, og stemmer overens med formlene i læreboka (kap. 13–18).

Bootstrap-eksemplene og ensidig ANOVA er flyttet til en egen notatbok: **`bootstrap_og_anova.ipynb`**.

In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

np.random.seed(2026)  # for reproduserbare bootstrap-resultater
plt.rcParams["figure.figsize"] = (6, 3.5)

---
## Oppgave 14.3

> I en undersøkelse regner man ut et konfidensintervall der man antar $\sigma = 4$.
> Hvor stor utvalgsstørrelse må man ha for at et 95 % konfidensintervall skal ha
> lengde mindre enn 1?

**Metode.** Bredden på et 95 % konfidensintervall for $\mu$ når $\sigma$ er kjent er

$$ \text{bredde} = 2 z^{*} \frac{\sigma}{\sqrt n}. $$

Vi løser $2 z^{*}\sigma/\sqrt n < 1$ med hensyn på $n$.

In [2]:
sigma = 4
z_star = stats.norm.ppf(0.975)          # z* for 95 % konfidensnivå
n_min = (2 * z_star * sigma / 1) ** 2   # bredde < 1
n = int(np.ceil(n_min))

print(f"z*          = {z_star:.4f}")
print(f"n > {n_min:.3f}")
print(f"Minste utvalgsstørrelse: n = {n}")

z*          = 1.9600
n > 245.853
Minste utvalgsstørrelse: n = 246


**Svar:** Vi trenger minst $n = 246$ observasjoner.

---
## Oppgave 15.1

> Anta et tilfeldig utvalg og regn ut $p$-verdien til den oppgitte testen, og
> oppsummer i hvilken grad $H_0$ virker urimelig.
>
> **a.** $\bar x_{obs}=0.5,\ \sigma=1,\ n=100$. Test $H_0:\mu=0$ mot $H_A:\mu\neq 0$.
>
> **b.** $\bar x_{obs}=0.5,\ \sigma=1,\ n=10$. Test $H_0:\mu=0$ mot $H_A:\mu\neq 0$.
>
> **c.** $\bar x_{obs}=6.7,\ \sigma=1,\ n=400$. Test $H_0:\mu=5$ mot $H_A:\mu\neq 5$.

In [3]:
def z_test_p_value(xbar, sigma, n, mu0):
    # Tosidig z-test for et gjennomsnitt naar sigma er kjent.
    T_obs = (xbar - mu0) / (sigma / np.sqrt(n))
    p_value = 2 * stats.norm.sf(abs(T_obs))
    return T_obs, p_value

oppgaver_15_1 = {
    "a": dict(xbar=0.5, sigma=1, n=100, mu0=0),
    "b": dict(xbar=0.5, sigma=1, n=10,  mu0=0),
    "c": dict(xbar=6.7, sigma=1, n=400, mu0=5),
}

for label, kw in oppgaver_15_1.items():
    T_obs, p_value = z_test_p_value(**kw)
    print(f"{label}) T_obs = {T_obs:8.4f}   p-verdi = {p_value:.6g}")

a) T_obs =   5.0000   p-verdi = 5.73303e-07
b) T_obs =   1.5811   p-verdi = 0.113846
c) T_obs =  34.0000   p-verdi = 2.2278e-253


**Tolkning.**

- **a)** $T_{obs}=5.00$, $p\approx 5.7\times10^{-7}$: ekstremt liten $p$-verdi $\Rightarrow$ $H_0$ virker svært urimelig.
- **b)** $T_{obs}=1.58$, $p\approx 0.114$: $p>0.05$ $\Rightarrow$ $H_0$ virker *ikke* urimelig på vanlig nivå.
- **c)** $T_{obs}=34.0$, $p\approx 2.2\times10^{-253}$: praktisk talt umulig under $H_0$ $\Rightarrow$ $H_0$ virker ekstremt urimelig.

---
## Oppgave 15.2

> Ved hjelp av et tilfeldig utvalg med $n=500$ tester vi $H_0:\mu=0$ mot
> $H_A:\mu\neq 0$ og får en $p$-verdi på 5 %. Hva er verdien til testobservatoren
> $T_{obs}$ hvis vi vet at $\bar x>0$?

In [4]:
p_value = 0.05
T_abs = stats.norm.ppf(1 - p_value / 2)  # løser 2*P(Z <= -|T|) = p_value

print(f"|T_obs| = {T_abs:.4f}")
# Siden xbar > 0 må testobservatoren selv være positiv:
T_obs = T_abs
print(f"T_obs   = {T_obs:.2f}")

|T_obs| = 1.9600
T_obs   = 1.96


**Svar:** $T_{obs} = 1.96$ (positiv, siden $\bar x>0$).

---
## Oppgave 15.3

> Vi er interessert i å teste $H_0:\mu=\mu_0$ mot $H_A:\mu\neq\mu_0$. Anta du er
> gitt følgende 95 % konfidensintervaller for $\mu$. Kan du forkaste $H_0$ på et
> 5 % nivå?
>
> **a.** 95 % KI $=[0.5,\,6]$ og $\mu_0=0$.
>
> **b.** 95 % KI $=[-0.2,\,3]$ og $\mu_0=0$.
>
> **c.** 95 % KI $=[-100.5,\,255]$ og $\mu_0=100$.

**Metode.** Et 95 % konfidensintervall og en tosidig test på $\alpha=5\,\%$ gir
samme konklusjon: forkast $H_0$ hvis og bare hvis $\mu_0$ ligger *utenfor*
intervallet.

In [5]:
cases_15_3 = {
    "a": dict(lo=0.5, hi=6, mu0=0),
    "b": dict(lo=-0.2, hi=3, mu0=0),
    "c": dict(lo=-100.5, hi=255, mu0=100),
}

for label, kw in cases_15_3.items():
    inside = kw["lo"] <= kw["mu0"] <= kw["hi"]
    beslutning = "BEHOLD H0" if inside else "FORKAST H0"
    print(f"{label}) KI = [{kw['lo']}, {kw['hi']}], mu0 = {kw['mu0']} "
          f"-> mu0 {'i' if inside else 'IKKE i'} intervallet -> {beslutning}")

a) KI = [0.5, 6], mu0 = 0 -> mu0 IKKE i intervallet -> FORKAST H0
b) KI = [-0.2, 3], mu0 = 0 -> mu0 i intervallet -> BEHOLD H0
c) KI = [-100.5, 255], mu0 = 100 -> mu0 i intervallet -> BEHOLD H0


**Svar:** a) forkast $H_0$.  b) behold $H_0$.  c) behold $H_0$.

---
## Oppgave 16.4

> Et konfidensintervall for $\mu$ basert på et tilfeldig utvalg av $n=22$
> observasjoner er $(99.766,\ 101.234)$.
>
> **a.** Finn $\bar x$.  **b.** Finn feilmarginen.
> **c.** Standardavviket i utvalget er $s=2.0$. Hva er konfidensnivået til intervallet?

In [6]:
lo, hi = 99.766, 101.234
n = 22
s = 2.0

xbar = (lo + hi) / 2
feilmargin = (hi - lo) / 2

df = n - 1
t_star = feilmargin / (s / np.sqrt(n))
konfidensniva = 1 - 2 * stats.t.sf(t_star, df)

print(f"a) xbar        = {xbar:.4f}")
print(f"b) feilmargin  = {feilmargin:.4f}")
print(f"c) t*          = {t_star:.4f}  (df = {df})")
print(f"   konfidensnivå = {konfidensniva*100:.2f} %")

a) xbar        = 100.5000
b) feilmargin  = 0.7340
c) t*          = 1.7214  (df = 21)
   konfidensnivå = 90.01 %


**Svar:** $\bar x = 100.500$, feilmargin $= 0.734$, og konfidensnivået er **90 %** ($t^{*}=1.721$ med $df=21$).

---
## Oppgave 16.5

> Det påstås at gjennomsnittsalderen til folk som stemmer KrF er over 50 år. For
> å teste dette tas et tilfeldig utvalg på $n=287$ KrF-velgere. Snittalderen i
> utvalget er $\bar x=51.3$ år, med et standardavvik på $s=14.2$ år.
> Signifikansnivået er $\alpha=0.05$.
>
> **a.** Skriv opp $H_0$ og $H_A$.  **b.** Beregn testobservatoren $t$.
> **c.** Beregn $p$-verdien.  **d.** Har vi tilstrekkelig støtte for $H_A$?

In [7]:
xbar, s, n, mu0 = 51.3, 14.2, 287, 50

T_obs = (xbar - mu0) / (s / np.sqrt(n))
df = n - 1
p_value = stats.t.sf(T_obs, df)   # ensidig test: HA: mu > 50

print("a) H0: mu = 50      HA: mu > 50")
print(f"b) T_obs = {T_obs:.4f}   (df = {df})")
print(f"c) p-verdi = {p_value:.4f}")
print(f"d) {'IKKE forkast H0' if p_value > 0.05 else 'Forkast H0'} "
      f"(p {'>' if p_value>0.05 else '<'} 0.05)")

a) H0: mu = 50      HA: mu > 50
b) T_obs = 1.5509   (df = 286)
c) p-verdi = 0.0610
d) IKKE forkast H0 (p > 0.05)


**Svar:** $T_{obs}=1.551$, $p\approx0.061>0.05$ $\Rightarrow$ vi har **ikke** tilstrekkelig statistisk støtte for $H_A$.

---
## Oppgave 16.6

> La oss anta at vekten på innholdet i en pakke cornflakes er normalfordelt, og
> at vi har trukket et tilfeldig utvalg på syv pakker. På pakkene står det 500 g.
> Vi veier pakkene og noterer vektene: 502, 498, 479, 492, 488, 494, 494.
>
> Vi ønsker å teste om pakkene i gjennomsnitt virkelig inneholder 500 g.
> Signifikansnivået settes til $\alpha=0.05$.
>
> **a.** Skriv opp $H_0$ og $H_A$.  **b.** Beregn testobservatoren $t$.
> **c.** Finn kritisk $t$-verdi eller regn ut eksakt $p$-verdi.  **d.** Kan vi beholde $H_0$?

In [8]:
vekter = np.array([502, 498, 479, 492, 488, 494, 494])
n = len(vekter)
xbar = vekter.mean()
s = vekter.std(ddof=1)   # utvalgsstandardavvik (Bessel-korrigert)
mu0 = 500

T_obs = (xbar - mu0) / (s / np.sqrt(n))
df = n - 1
t_krit = stats.t.ppf(0.975, df)
p_value = 2 * stats.t.sf(abs(T_obs), df)

print("a) H0: mu = 500      HA: mu != 500")
print(f"   xbar = {xbar:.4f} g,  s = {s:.4f} g")
print(f"b) T_obs = {T_obs:.4f}   (df = {df})")
print(f"c) t_krit(0.025, {df}) = {t_krit:.4f},   p-verdi = {p_value:.4f}")
print(f"d) {'Behold H0' if p_value > 0.05 else 'Forkast H0'}")

a) H0: mu = 500      HA: mu != 500
   xbar = 492.4286 g,  s = 7.3905 g
b) T_obs = -2.7105   (df = 6)
c) t_krit(0.025, 6) = 2.4469,   p-verdi = 0.0351
d) Forkast H0


**Svar:** $\bar x\approx492.43$ g, $s\approx7.39$ g, $T_{obs}\approx-2.71$, $p\approx0.035<0.05$ $\Rightarrow$ vi **forkaster** $H_0$: pakkene inneholder i snitt signifikant mindre enn 500 g.

---
## Oppgave 17.5

> I denne oppgaven skal vi bruke utvalget på $n=295$ fra undersøkelsen «BItrent»
> til å undersøke om andelen i populasjonen som forventer å få karakteren B
> eller bedre på eksamen, er over 50 % (0.5). Av de 295 i utvalget var det 152
> personer som svarte at de forventet å få karakteren B eller bedre.
>
> **a.** Beregn et estimat $\hat p$.
> **b.** Skriv opp $H_0$ og $H_A$ (signifikansnivå $\alpha=0.05$).
> **c.** Beregn testobservatoren.  **d.** Beregn testens $p$-verdi.
> **e.** Konkluder på testen i et lettfattelig språk.

In [9]:
x, n = 152, 295
p0 = 0.5

phat = x / n
T_obs = (phat - p0) / np.sqrt(p0 * (1 - p0) / n)
p_value = stats.norm.sf(T_obs)   # ensidig test: HA: p > 0.5

print(f"a) phat = {phat:.4f}")
print("b) H0: p = 0.5      HA: p > 0.5")
print(f"c) T_obs = {T_obs:.4f}")
print(f"d) p-verdi = {p_value:.4f}")
print(f"e) {'Forkast H0' if p_value < 0.05 else 'Ikke forkast H0'}")

a) phat = 0.5153
b) H0: p = 0.5      HA: p > 0.5
c) T_obs = 0.5240
d) p-verdi = 0.3001
e) Ikke forkast H0


**Svar:** $\hat p=0.5153$, $T_{obs}=0.524$, $p\approx0.300>0.05$ $\Rightarrow$ vi har **ikke** nok evidens til å si at andelen er over 50 %.

---
## Oppgave 17.6

> I en nasjonal undersøkelse «Røyk forgifter» i USA ble det samlet inn data på
> 14 variabler relatert til kvinners fødsler. I alt ble det samlet inn data på
> $n=1388$ fødsler. I utvalget røyket 212 kvinner under graviditeten.
>
> **a.** Beregn et estimat $\hat p$ for andelen kvinner i populasjonen som røyker
> under graviditet.
> **b.** Beregn et 95 % konfidensintervall for andelen $p$.
> **c.** Gi en tolkning av konfidensintervallet.
> **d.** Det påstås at andelen er over 14 %. Test dette på $\alpha=0.05$.
> **e.–f.** Beregn testobservator og $p$-verdi.  **g.** Konkluder.

In [10]:
x, n = 212, 1388

phat = x / n
se = np.sqrt(phat * (1 - phat) / n)
z_star = stats.norm.ppf(0.975)
ki = (phat - z_star * se, phat + z_star * se)

p0 = 0.14
T_obs = (phat - p0) / np.sqrt(p0 * (1 - p0) / n)
p_value = stats.norm.sf(T_obs)   # ensidig test: HA: p > 0.14

print(f"a) phat = {phat:.4f}")
print(f"b) 95% KI for p: [{ki[0]:.4f}, {ki[1]:.4f}]")
print("d) H0: p = 0.14      HA: p > 0.14")
print(f"e) T_obs = {T_obs:.4f}")
print(f"f) p-verdi = {p_value:.4f}")
print(f"g) {'Forkast H0' if p_value < 0.05 else 'Ikke forkast H0'}")

a) phat = 0.1527
b) 95% KI for p: [0.1338, 0.1717]
d) H0: p = 0.14      HA: p > 0.14
e) T_obs = 1.3676
f) p-verdi = 0.0857
g) Ikke forkast H0


**Svar:**

- **a–b)** $\hat p = 0.153$, 95 % KI $\approx [0.134,\ 0.172]$.
- **c)** Vi er 95 % sikre på at andelen gravide som røyker i populasjonen ligger mellom ca. 13.4 % og 17.2 %.
- **d–g)** $T_{obs}=1.368$, $p\approx0.086>0.05$ $\Rightarrow$ vi har **ikke** tilstrekkelig evidens for at andelen er over 14 %.

---
## Oppgave 18.1

> Tilfeldig utvalgte studenter som tok et matematikkurs, ble spurt om hvor enige
> de var i påstanden *«Jeg er sjelden stresset i matematikkundervisningen»*
> (skala 1–5). Av de 106 kvinnene som deltok, var gjennomsnittet 2.97 og
> standardavviket 1.24, mens blant de 115 mennene var gjennomsnittet 3.26 og
> standardavviket 1.20.
>
> **a.** Lag et 95 % konfidensintervall for differansen $\mu_M-\mu_K$ mellom
> populasjonsgjennomsnittene til menn og kvinner.
> **b.** Test påstanden om at menn er mindre stresset i matematikkundervisningen
> enn kvinner. Bruk $\alpha=0.05$.

In [11]:
n_m, xbar_m, s_m = 115, 3.26, 1.20   # menn
n_k, xbar_k, s_k = 106, 2.97, 1.24   # kvinner

diff = xbar_m - xbar_k
se = np.sqrt(s_m**2 / n_m + s_k**2 / n_k)

# Welch-Satterthwaite frihetsgrader (samme som en kalkulator/statistikkpakke gir)
df = (s_m**2/n_m + s_k**2/n_k)**2 / (
      (s_m**2/n_m)**2 / (n_m - 1) + (s_k**2/n_k)**2 / (n_k - 1))
t_star = stats.t.ppf(0.975, df)
ki = (diff - t_star * se, diff + t_star * se)

T_obs = diff / se
p_value = stats.t.sf(T_obs, df)   # ensidig: HA: mu_M > mu_K (mindre stresset = høyere skår)

print(f"a) diff (M - K) = {diff:.4f}")
print(f"   SE = {se:.4f},  df (Welch) = {df:.2f},  t* = {t_star:.4f}")
print(f"   95% KI for mu_M - mu_K: [{ki[0]:.4f}, {ki[1]:.4f}]")
print()
print("b) H0: mu_M = mu_K      HA: mu_M > mu_K")
print(f"   T_obs = {T_obs:.4f}")
print(f"   p-verdi = {p_value:.5f}")
print(f"   {'Forkast H0' if p_value < 0.05 else 'Ikke forkast H0'}")

a) diff (M - K) = 0.2900
   SE = 0.1644,  df (Welch) = 216.16,  t* = 1.9710
   95% KI for mu_M - mu_K: [-0.0340, 0.6140]

b) H0: mu_M = mu_K      HA: mu_M > mu_K
   T_obs = 1.7640
   p-verdi = 0.03957
   Forkast H0


**Svar:** 95 % KI for $\mu_M-\mu_K \approx [-0.034,\ 0.614]$ (dekker 0, men såvidt). Ensidig test: $T_{obs}=1.764$, $p\approx0.040<0.05$ $\Rightarrow$ vi **forkaster** $H_0$ — det er (svak, men signifikant) støtte for at menn rapporterer å være mindre stresset enn kvinner i matematikkundervisningen.

---
## Oppgave 18.2

> Respons Analyse foretok i 2015 en landsdekkende spørreundersøkelse ved hjelp
> av telefonintervju med totalt 2001 respondenter. Vi antar at utvalget var
> tilfeldig. Vi ser på to populasjoner: de som bruker Facebook/Twitter daglig, og
> de som ikke bruker Facebook/Twitter daglig. Det var 1249 respondenter som
> brukte Facebook/Twitter daglig (gruppe 1), med gjennomsnittsalder 41.3 år og
> standardavvik 15.7 år. De øvrige 752 respondentene (gruppe 2) hadde
> gjennomsnittsalder 59.3 år og standardavvik 16.4 år.
>
> Du skal teste påstanden om at daglige brukere av Facebook/Twitter er yngre enn
> personer som ikke bruker Facebook/Twitter daglig.
>
> **a.** Skriv opp hypotesene.  **b.** Regn ut testobservatoren. Er betingelsene
> for å utføre testen oppfylt?  **c.** Bestem $p$-verdien.
> **d.** Utfør hypotesetesten med signifikansnivå $\alpha=0.01$.

In [12]:
n1, xbar1, s1 = 1249, 41.3, 15.7   # daglige brukere
n2, xbar2, s2 = 752,  59.3, 16.4   # ikke daglige brukere

se = np.sqrt(s1**2 / n1 + s2**2 / n2)
T_obs = (xbar1 - xbar2) / se

df = (s1**2/n1 + s2**2/n2)**2 / (
      (s1**2/n1)**2 / (n1 - 1) + (s2**2/n2)**2 / (n2 - 1))
p_value = stats.t.cdf(T_obs, df)   # ensidig, venstre hale: HA: mu1 < mu2

print("a) H0: mu1 = mu2      HA: mu1 < mu2")
print(f"b) T_obs = {T_obs:.4f}   (df = {df:.1f})")
print("   Betingelser: begge utvalg er store (n1, n2 >> 30), sentralgrensesetningen")
print("   sikrer at gjennomsnittene er tilnærmet normalfordelte -> testen er gyldig.")
print(f"c) p-verdi = {p_value:.4e}")
print(f"d) {'Forkast H0' if p_value < 0.01 else 'Ikke forkast H0'} (alpha = 0.01)")

a) H0: mu1 = mu2      HA: mu1 < mu2
b) T_obs = -24.1614   (df = 1528.4)
   Betingelser: begge utvalg er store (n1, n2 >> 30), sentralgrensesetningen
   sikrer at gjennomsnittene er tilnærmet normalfordelte -> testen er gyldig.
c) p-verdi = 8.3702e-110
d) Forkast H0 (alpha = 0.01)


**Svar:** $T_{obs}\approx-24.2$, $p\approx8.4\times10^{-110}$, som er langt mindre enn $\alpha=0.01$ $\Rightarrow$ vi **forkaster** $H_0$: det er overveldende evidens for at daglige Facebook/Twitter-brukere er yngre i gjennomsnitt enn de som ikke bruker det daglig.

---
## Oppsummering av alle svar

| Oppgave | Nøkkeltall | Konklusjon |
|---|---|---|
| 14.3 | $n>245.9$ | $n=246$ |
| 15.1 a/b/c | $T=5.00\,/\,1.58\,/\,34.0$ | sterkt urimelig / ikke urimelig / ekstremt urimelig |
| 15.2 | $\lvert T_{obs}\rvert=1.96$ | $T_{obs}=1.96$ |
| 15.3 a/b/c | — | forkast / behold / behold |
| 16.4 | $t^{*}=1.721,\ df=21$ | 90 % konfidensnivå |
| 16.5 | $p\approx0.061$ | ikke forkast $H_0$ |
| 16.6 | $p\approx0.035$ | forkast $H_0$ |
| 17.5 | $p\approx0.300$ | ikke forkast $H_0$ |
| 17.6 | $p\approx0.086$ | ikke forkast $H_0$ |
| 18.1 | $p\approx0.040$ | forkast $H_0$ |
| 18.2 | $p\approx8\times10^{-110}$ | forkast $H_0$ |